# BRCA1 ESM-DMS Experimental Pipeline

This notebook runs the class-based BRCA1 workflow from `data/mavedb_data`: MaveDB counts and functional scores are keyed by `hgvs_nt`, mutated protein sequences are reconstructed from the BRCA1 wildtype protein reference where the nucleotide mutation has an unambiguous amino-acid consequence, embeddings are pooled and cached, DeltaSAE features are inferred, and model fitness is compared with MaveDB functional scores.

In [1]:
from pathlib import Path
import os
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break

os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from esmDMS import CellularDMSInput, ESMDMSConfig, esmDMS

DATA_DIR = REPO_ROOT / "data" / "mavedb_data"
ANALYSIS_DIR = REPO_ROOT / "data" / "esm_data_analysis" / "BRCA1_experimental"
SEQUENCE_DIR = ANALYSIS_DIR / "sequence_data"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

for directory in (SEQUENCE_DIR, FIGURE_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid")
REPO_ROOT

PosixPath('/Users/dylanwells/popDMS/dylan_refactor')

## Configure BRCA1

`SAE_EMBEDDING_TYPE` selects which saved pooled embedding cache trains or feeds the sparse autoencoder. Use `"mean_pool"` or `"max_pool"`. `EMBEDDING_MODEL` can be an ESM-2 Hugging Face model such as `"facebook/esm2_t6_8M_UR50D"` or an ESMC model identifier supported by the local environment.

In [2]:
BRCA1_INPUT = CellularDMSInput(
    reference_nuc_path=DATA_DIR / "BRCA1_reference_sequence.dat",
    mavedb_csv_path=DATA_DIR / "BRCA1_counts.csv",
    scores_csv_path=DATA_DIR / "BRCA1_scores.csv",
    reference_kind="protein",
    primary_key="hgvs_nt",
)

EMBEDDING_MODEL = "biohub/ESMC-300M"
REPRESENTATIVE_LAYER = 24
SAE_EMBEDDING_TYPE = "max_pool"  # "mean_pool" or "max_pool"
ABSTRACTION_METHOD = "DeltaSAE"
NORM_SCHEME = "none"

SAE_PARAMS = {
    "n_features": 10000,
    "sparsity_coeff": 1e-3,
    "sparsity_mode": "batchtopk",  # "normal", "topk", or "batchtopk"
    "k": 200,
    "epochs": 200,
    "batch_size": 64,
    "train_frac": 0.8,
    "lr": 1e-3,
    "seed": 42,
    "run_label": f"{SAE_EMBEDDING_TYPE}_batchtopk200_12800feat",
    "norm_scheme": NORM_SCHEME,
    # "pretrained_model_path": SEQUENCE_DIR / "sae_models" / "existing_model.pt",
}

config = ESMDMSConfig(
    embedding_model=EMBEDDING_MODEL,
    embedding_type=SAE_EMBEDDING_TYPE,
    local_or_disk="both",
    save_dir=str(SEQUENCE_DIR),
    dataset_name="BRCA1",
)

runner = esmDMS(input_data=BRCA1_INPUT, config=config)
runner

## Process Counts And Scores

The parser keeps `hgvs_nt` as `SequenceIndex`, stores a separate key-to-protein-sequence map, loads functional scores keyed by `hgvs_nt`, and skips rows that cannot produce a unique protein sequence from a protein-only reference, such as intronic HGVS entries or ambiguous codon effects.

In [3]:
runner.process_raw_data(drop_stop_codons=True)

processing_summary = pd.DataFrame([{
    "dataset": "BRCA1",
    "reference_kind": runner.reference_kind,
    "count_rows": len(runner.sequence_dataframe),
    "mutation_keys_in_counts": runner.sequence_dataframe["SequenceIndex"].nunique(),
    "protein_sequence_keys": len(runner.sequence_to_protein_sequence),
    "replicates": runner.sequence_dataframe["Replicate"].nunique(),
    "generations": sorted(runner.sequence_dataframe["Generation"].unique()),
    "score_rows": len(runner.scores_dataframe),
    "skipped_counts": runner.sequence_metadata.attrs.get("skipped_counts", {}),
}])
processing_summary.to_csv(TABLE_DIR / "BRCA1_processing_summary.csv", index=False)
processing_summary

,dataset,reference_kind,count_rows,mutation_keys_in_counts,protein_sequence_keys,replicates,generations,score_rows,skipped_counts
0,BRCA1,protein,12624,2104,2105,2,"[0, 1, 2]",3893,{'ambiguous_protein_effect_from_protein_refere...


In [4]:
runner.sequence_dataframe.head()

,SequenceIndex,hgvs_nt,Replicate,ReplicateName,Generation,Frequency,CountColumn
0,NM_007294.3:c.5565A>T,NM_007294.3:c.5565A>T,1,rep1,0,476.0,count_library
1,NM_007294.3:c.5565A>T,NM_007294.3:c.5565A>T,1,rep1,1,2880.0,count_day5_rep1
2,NM_007294.3:c.5565A>T,NM_007294.3:c.5565A>T,1,rep1,2,2635.0,count_day11_rep1
3,NM_007294.3:c.5565A>T,NM_007294.3:c.5565A>T,2,rep2,0,476.0,count_library
4,NM_007294.3:c.5565A>T,NM_007294.3:c.5565A>T,2,rep2,1,2935.0,count_day5_rep2


In [5]:
runner.scores_dataframe.head()

,accession,hgvs_nt,score,score_rep1,score_rep2,score_rna,score_rna_rep1,score_rna_rep2,SequenceIndex
0,urn:mavedb:00000097-0-2#1,NM_007294.3:c.5565A>T,-0.015322,-0.116153,0.085509,-0.403450,-0.160708,-0.482648,NM_007294.3:c.5565A>T
1,urn:mavedb:00000097-0-2#2,NM_007294.3:c.5565A>G,0.021941,0.174501,-0.130620,-0.289526,0.404663,-1.054867,NM_007294.3:c.5565A>G
2,urn:mavedb:00000097-0-2#3,NM_007294.3:c.5565A>C,0.231183,0.151333,0.311032,0.207660,0.410202,0.168734,NM_007294.3:c.5565A>C
3,urn:mavedb:00000097-0-2#4,NM_007294.3:c.5564T>G,-0.464328,-0.140845,-0.787812,0.343402,1.098202,-0.569539,NM_007294.3:c.5564T>G
4,urn:mavedb:00000097-0-2#5,NM_007294.3:c.5564T>C,-0.291519,-0.477009,-0.106029,0.303770,0.111163,0.567510,NM_007294.3:c.5564T>C


## Embed Mutated And Wildtype Proteins

Embedding writes `mean_pool`, `max_pool`, and `per_residue` caches for each layer. The wildtype protein is embedded even though it does not appear as a count row, because DeltaSAE subtracts its SAE representation.

In [6]:
RUN_LOCAL_EMBEDDINGS = False
CREATE_EMBEDDING_JOB = False
SUBMIT_JOBS = False

if RUN_LOCAL_EMBEDDINGS:
    runner.embed_all_sequences(layer=REPRESENTATIVE_LAYER, test_num=10)

if CREATE_EMBEDDING_JOB:
    embedding_job = runner.create_embedding_batch_job(
        job_dir=SEQUENCE_DIR / "embedding_batch_jobs",
        n_chunks=40,
        max_active_jobs=4,
        job_name="brca1_esm_embed",
        partition="any_cpu",
        mem="24G",
        time="08:00:00",
        python_executable="python3",
        scratch_root="/scr",
        submit=SUBMIT_JOBS,
    )
    display(pd.DataFrame([{
        "script_path": str(embedding_job["script_path"]),
        "payload_path": str(embedding_job["payload_path"]),
        "job_id": embedding_job["job_id"],
    }]))

In [7]:
MERGE_EMBEDDING_OUTPUTS = False

if MERGE_EMBEDDING_OUTPUTS:
    runner.merge_embedding_batch_outputs(
        job_dir=SEQUENCE_DIR / "embedding_batch_jobs",
        layer="all",
        save_layers=True,
    )

cache_status = pd.DataFrame([
    {
        "layer": REPRESENTATIVE_LAYER,
        "embedding_type": embedding_type,
        "path": str(runner._embedding_path(REPRESENTATIVE_LAYER, embedding_type)),
        "exists": runner._embedding_path(REPRESENTATIVE_LAYER, embedding_type).exists(),
    }
    for embedding_type in ("mean_pool", "max_pool", "per_residue")
])
cache_status.to_csv(TABLE_DIR / "BRCA1_embedding_cache_status.csv", index=False)
cache_status

,layer,embedding_type,path,exists
0,24,mean_pool,/Users/dylanwells/popDMS/dylan_refactor/data/e...,True
1,24,max_pool,/Users/dylanwells/popDMS/dylan_refactor/data/e...,True
2,24,per_residue,/Users/dylanwells/popDMS/dylan_refactor/data/e...,True


## Train Or Load DeltaSAE Features

`create_feature_space(..., method="DeltaSAE")` trains the selected SAE variant on the pooled embeddings, keeps SAE dimensions with activation frequency strictly between 0 and 1, and returns `SAE(mutant) - SAE(wildtype)` for every `hgvs_nt` mutation key. Set `FORCE_RECOMPUTE_DELTA_SAE = True` to bypass the cached DeltaSAE feature file.

In [9]:
RUN_DELTA_SAE = True
FORCE_RECOMPUTE_DELTA_SAE = False

if RUN_DELTA_SAE:
    delta_sae_features = runner.create_feature_space(
        layer=REPRESENTATIVE_LAYER,
        method=ABSTRACTION_METHOD,
        method_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
        force_recompute=FORCE_RECOMPUTE_DELTA_SAE,
    )
    print(f"DeltaSAE feature count: {len(delta_sae_features)}")
    print(f"DeltaSAE dimensions: {len(next(iter(delta_sae_features.values())))}")

DeltaSAE feature count: 2104
DeltaSAE dimensions: 4


In [15]:
PLOT_SAE_QA = True

if PLOT_SAE_QA:
    runner.visualize_sae_reconstructions(
        layer=REPRESENTATIVE_LAYER,
        method_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_reconstruction_QA.png",
    )
    plt.show()

FileNotFoundError: No SAE visualisation data found at /Users/dylanwells/popDMS/dylan_refactor/data/esm_data_analysis/BRCA1_experimental/sequence_data/sae_models/BRCA1_max_pool_sae_Layer_24_10000_0.001_batchtopk_k200_max_pool_batchtopk200_12800feat_viz_data.pkl. Run create_feature_space(..., method='SAE') first.

## Infer Selection On DeltaSAE Features

The inferred model fitness is `1 + sel_coeffs_joint dot DeltaSAE`.

In [ ]:
RUN_INFERENCE = False
LOAD_COMPLETED_INFERENCE = False
inference_result = None

if RUN_INFERENCE:
    inference_result = runner.run_feature_inference(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        abstraction_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
    )
elif LOAD_COMPLETED_INFERENCE:
    inference_result = runner.load_inference_results(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        embedding_type=SAE_EMBEDDING_TYPE,
    )

if inference_result is not None:
    inference_summary = pd.DataFrame([{
        "layer": REPRESENTATIVE_LAYER,
        "embedding_type": SAE_EMBEDDING_TYPE,
        "abstraction_method": ABSTRACTION_METHOD,
        "n_replicates": inference_result.s.shape[0],
        "n_dimensions": inference_result.s.shape[1],
        "gamma_opt": inference_result.gamma_opt,
        "s_joint_mean": inference_result.s_joint.mean(),
        "s_joint_std": inference_result.s_joint.std(),
    }])
    inference_summary.to_csv(TABLE_DIR / "BRCA1_DeltaSAE_inference_summary.csv", index=False)
    display(inference_summary)
else:
    print("Set RUN_INFERENCE or LOAD_COMPLETED_INFERENCE to True after DeltaSAE features are available.")

## Regularization Diagnostics

In [ ]:
PLOT_REGULARIZATION = False

if PLOT_REGULARIZATION:
    fig, regularization_df = runner.plot_regularization_curve(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        abstraction_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_regularization.png",
    )
    regularization_df.to_csv(TABLE_DIR / "BRCA1_DeltaSAE_regularization.csv", index=False)
    plt.show()

## Compare Inferred Fitness With MaveDB Scores

In [ ]:
PLOT_SCORE_COMPARISON = False

if PLOT_SCORE_COMPARISON:
    fig, score_comparison_df, score_stats = runner.plot_functional_score_comparison(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        abstraction_params=SAE_PARAMS,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        score_col="score",
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_fitness_vs_score.png",
    )
    score_comparison_df.to_csv(TABLE_DIR / "BRCA1_DeltaSAE_fitness_vs_scores.csv", index=False)
    display(pd.DataFrame([score_stats]))
    plt.show()

## Cross-Replicate Consistency

In [ ]:
PLOT_REPLICATE_CONSISTENCY = False

if PLOT_REPLICATE_CONSISTENCY:
    runner.plot_rep_sel_comps(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        label="BRCA1 DeltaSAE",
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_selection_replicate_scatter.png",
    )
    plt.show()

    runner.plot_rep_fit_comps(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        label="BRCA1 DeltaSAE",
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_fitness_replicate_scatter.png",
    )
    plt.show()

## Selection Coefficient Distribution

In [ ]:
PLOT_SELECTION_COEFFICIENTS = False

if PLOT_SELECTION_COEFFICIENTS:
    fig, coef_df = runner.plot_selection_coefficient_distribution(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        embedding_type=SAE_EMBEDDING_TYPE,
        norm_scheme=NORM_SCHEME,
        output_path=FIGURE_DIR / f"BRCA1_{SAE_EMBEDDING_TYPE}_DeltaSAE_selection_coefficients.png",
    )
    coef_df.to_csv(TABLE_DIR / "BRCA1_DeltaSAE_selection_coefficients.csv", index=False)
    plt.show()